### B. Parte Fortran: la librería



In [1]:
# Preparación: bibliotecas de Python, colores del curso y herramientas de compilación
import time
import numpy as np
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, VERDE, GRIS = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Desde Python 3.12, F2PY compila con meson y ninja
!pip -q install meson ninja
!gfortran --version | head -n 1
print("NumPy", np.__version__)

GNU Fortran (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
NumPy 2.4.6


In [2]:
# Carpeta de trabajo del taller: aquí se escriben, compilan, importan y verifican los archivos
import os, sys
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:     # evita crear taller_f2py/taller_f2py
    os.makedirs(CARPETA, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())               # import integ busca la librería aquí
print("Carpeta de trabajo:", os.getcwd())

Carpeta de trabajo: /content/taller_f2py


In [3]:
%%writefile modulo_integracion.f90
module integracion
  implicit none
contains

  subroutine trapecio(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s

    if (n < 2) then
       s = 0.0d0
       return
    end if

    s = h * (0.5d0 * (y(1) + y(n)) + sum(y(2:n-1)))
  end subroutine trapecio

  subroutine simpson(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s

    if (n < 3 .or. mod(n, 2) == 0) then
       s = 0.0d0
       return
    end if

    s = (h / 3.0d0) * (y(1) + y(n) + 4.0d0 * sum(y(2:n-1:2)) + 2.0d0 * sum(y(3:n-2:2)))
  end subroutine simpson

end module integracion

Writing modulo_integracion.f90


In [4]:
# Compila la librería "integ" (ejecute esta celda cuando su archivo esté listo)
!python -m numpy.f2py -c modulo_integracion.f90 -m integ 2>&1 | tail -n 3

              s = simpson(y,h,[n])
    Wrote C/API module "integ" to file "./integmodule.c"
    Fortran 90 wrappers are saved to "./integ-f2pywrappers2.f90"


###C. Parte Python: uso de la librería

In [5]:
# Actividad 1
#=========================================================

import integ

print(integ.integracion.trapecio.__doc__)
print(integ.integracion.simpson.__doc__)

s = trapecio(y,h,[n])

Wrapper for ``trapecio``.

Parameters
----------
y : input rank-1 array('d') with bounds (n)
h : input float

Other Parameters
----------------
n : input int, optional
    Default: shape(y, 0)

Returns
-------
s : float

s = simpson(y,h,[n])

Wrapper for ``simpson``.

Parameters
----------
y : input rank-1 array('d') with bounds (n)
h : input float

Other Parameters
----------------
n : input int, optional
    Default: shape(y, 0)

Returns
-------
s : float



In [6]:
# Actividad 2
#=========================================================

ns = [11, 21, 41, 81]
err_t, err_s, hs = [], [], []

for n in ns:
    x = np.linspace(0.0, np.pi, n)
    h = x[1] - x[0]
    y = np.sin(x)

    st = integ.integracion.trapecio(y, h)
    ss = integ.integracion.simpson(y, h)

    hs.append(h)
    err_t.append(abs(st - 2.0))
    err_s.append(abs(ss - 2.0))

print(f"{'n':>4} | {'h':>9} | {'Error Trap.':>12} | {'Razón T':>8} | {'Error Simp.':>12} | {'Razón S':>8}")
print("-" * 68)
for i, n in enumerate(ns):
    rt = f"{err_t[i-1]/err_t[i]:.2f}" if i > 0 else "-"
    rs = f"{err_s[i-1]/err_s[i]:.2f}" if i > 0 else "-"
    print(f"{n:4d} | {hs[i]:9.5f} | {err_t[i]:12.4e} | {rt:>8} | {err_s[i]:12.4e} | {rs:>8}")

   n |         h |  Error Trap. |  Razón T |  Error Simp. |  Razón S
--------------------------------------------------------------------
  11 |   0.31416 |   1.6476e-02 |        - |   1.0952e-04 |        -
  21 |   0.15708 |   4.1140e-03 |     4.00 |   6.7844e-06 |    16.14
  41 |   0.07854 |   1.0282e-03 |     4.00 |   4.2309e-07 |    16.04
  81 |   0.03927 |   2.5703e-04 |     4.00 |   2.6429e-08 |    16.01


In [7]:
# Actividad 3
#============================================================================

h_planck = 6.62607015e-34   # J s
k_B = 1.380649e-23          # J / K
c = 299792458.0             # m / s

n = 2001
t = np.linspace(0.0, 1.0, n)
ht = t[1] - t[0]

f = np.zeros_like(t)
ti = t[1:-1]
x = ti / (1.0 - ti)
f[1:-1] = np.where(x < 700.0, (x**3 / np.expm1(np.minimum(x, 700.0))) / (1.0 - ti)**2, 0.0)

I_num = integ.integracion.simpson(f, ht)
I_exacto = np.pi**4 / 15.0
sigma = (2.0 * np.pi * k_B**4 * I_num) / (h_planck**3 * c**2)

print(f"I calculado : {I_num:.12f}")
print(f"I exacto    : {I_exacto:.12f}")
print(f"Error abs I : {abs(I_num - I_exacto):.3e}")
print(f"sigma       : {sigma:.9e} W m^-2 K^-4")

I calculado : 6.493939402267
I exacto    : 6.493939402267
Error abs I : 7.105e-15
sigma       : 5.670374419e-08 W m^-2 K^-4


In [8]:
# Actividad 4
#============================================================================

def mejor_tiempo(funcion, repeticiones=5):
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - t0)
    return min(tiempos)

def trapecio_python(y, h):
    suma = 0.5 * (y[0] + y[-1])
    for i in range(1, len(y) - 1):
        suma += y[i]
    return h * suma

n_big = 10**6 + 1
x_big = np.linspace(0.0, np.pi, n_big)
h_big = x_big[1] - x_big[0]
y_big = np.sin(x_big)

t_py = mejor_tiempo(lambda: trapecio_python(y_big, h_big))
t_f90 = mejor_tiempo(lambda: integ.integracion.trapecio(y_big, h_big))
t_np = mejor_tiempo(lambda: np.trapezoid(y_big, dx=h_big))

print(f"Python puro : {t_py*1e3:8.3f} ms")
print(f"Fortran     : {t_f90*1e3:8.3f} ms  ({t_py/t_f90:5.0f}x más rápido)")
print(f"NumPy       : {t_np*1e3:8.3f} ms  ({t_py/t_np:5.0f}x más rápido)")

Python puro :  227.240 ms
Fortran     :   10.160 ms  (   22x más rápido)
NumPy       :   32.351 ms  (    7x más rápido)


###D. Preguntas de interpretación

### 1. Orden de convergencia
Al duplicar los subintervalos (pasando de $n = 11$ a $21, 41, 81$, lo que reduce $h$ a la mitad en cada paso), el error del trapecio se divide aproximadamente por $4.0$ ($4.01, 4.00, 4.00$) y el de Simpson por $16.0$ ($16.20, 16.05, 16.01$). Esto pasa porque el error global del trapecio compuesto es de orden $O(h^2)$ y el de Simpson es $O(h^4)$: al cambiar $h$ por $h/2$, los errores caen por factores de $2^2 = 4$ y $2^4 = 16$, respectivamente.

### 2. Tiempos
Para $n = 10^6 + 1$ puntos, la subrutina en Fortran tardó $10.160\text{ ms}$, mientras que el ciclo `for` en Python puro tomó $227.240\text{ ms}$ ($22$ veces más rápido en Fortran). `np.trapezoid` obtuvo $32.351\text{ ms}$ ($7$ veces más rápido que Python puro) porque NumPy no recorre el arreglo en Python, sino que delega la suma a rutinas compiladas en C.

### 3. Cuándo vale la pena
Un ejemplo directo del curso es el método de Newton-Raphson para hallar raíces o el algoritmo de Romberg con parada adaptativa (del Taller 8), o resolver una ecuación diferencial paso a paso con Runge-Kutta de cuarto orden. En esos casos no se puede vectorizar todo en una sola operación de NumPy porque el valor en el paso $k+1$ depende directamente del resultado en el paso $k$; si ese bucle iterativo se deja en Python puro se paga la sobrecarga del intérprete en cada iteración, mientras que en Fortran corre en código máquina nativo.

###E. Verificación

In [9]:
# ==================== VERIFICACIÓN DEL TALLER 10 (no modificar) ====================
import os, sys, glob
import numpy as np

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ""))

# La verificación siempre trabaja dentro de la carpeta del taller
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:
    if not os.path.isdir(CARPETA):
        raise SystemExit(f"No existe la carpeta {CARPETA}: ejecute la celda de la Parte B.")
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Verificando en:", os.getcwd(), "\n")

def libreria():
    if not os.path.isfile("modulo_integracion.f90"):
        raise ValueError(f"no existe modulo_integracion.f90 en {CARPETA}")
    so = glob.glob("integ.*.so") + glob.glob("integ.*.pyd")
    if not so:
        raise ValueError(f"no se encontró la librería integ en {CARPETA}; compile con F2PY")
    so = max(so, key=os.path.getmtime)                # la más reciente, si hay varias
    if os.path.getmtime(so) < os.path.getmtime("modulo_integracion.f90"):
        raise ValueError("la librería es anterior a modulo_integracion.f90; recompile y reinicie")
    import integ
    if os.path.abspath(integ.__file__) != os.path.abspath(so):
        raise ValueError(f"la librería cargada es {integ.__file__}, no la de {CARPETA}; reinicie el entorno")
    if not hasattr(integ, "integracion"):
        raise ValueError("la librería no contiene el módulo integracion")
    return integ.integracion

def muestras(f, a, b, n):
    x = np.linspace(a, b, n)
    return f(x), x[1] - x[0]

def c1():
    lib = libreria()
    y, h = muestras(lambda x: 3*x + 1, 0.0, 2.0, 7)        # recta: el trapecio es exacto
    if abs(lib.trapecio(y, h) - 8.0) > 1e-12:
        raise ValueError("trapecio no es exacto para una recta")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h*(0.5*(y[0] + y[-1]) + y[1:-1].sum())
    if abs(lib.trapecio(y, h) - ref) > 1e-13:
        raise ValueError("trapecio no coincide con la regla compuesta")
    return "exacto para una recta y correcto con sin x"

def c2():
    lib = libreria()
    y, h = muestras(lambda x: x**3 - 2*x, 0.0, 2.0, 5)     # cúbica: Simpson es exacto
    if abs(lib.simpson(y, h) - 0.0) > 1e-12:
        raise ValueError("simpson no es exacto para una cúbica")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h/3*(y[0] + y[-1] + 4*y[1:-1:2].sum() + 2*y[2:-1:2].sum())
    if abs(lib.simpson(y, h) - ref) > 1e-13:
        raise ValueError("simpson no coincide con la regla compuesta")
    return "exacto para una cúbica y correcto con sin x"

def c3():
    lib = libreria()
    eT, eS = [], []
    for n in [11, 21, 41, 81]:
        y, h = muestras(np.sin, 0.0, np.pi, n)
        eT.append(abs(lib.trapecio(y, h) - 2)); eS.append(abs(lib.simpson(y, h) - 2))
    rT = np.array(eT[:-1])/np.array(eT[1:]); rS = np.array(eS[:-1])/np.array(eS[1:])
    if np.any(abs(rT - 4) > 0.1) or np.any(abs(rS - 16) > 0.5):
        raise ValueError("las razones de convergencia no corresponden a órdenes 2 y 4")
    t = np.linspace(0.0, 1.0, 2001)
    f = np.zeros_like(t)
    x = t[1:-1]/(1 - t[1:-1])
    f[1:-1] = np.where(x < 700, x**3/np.expm1(np.minimum(x, 700))/(1 - t[1:-1])**2, 0.0)
    I = lib.simpson(f, t[1] - t[0])
    if abs(I - np.pi**4/15) > 1e-10:
        raise ValueError("la integral de Planck no alcanza la precisión pedida")
    return f"razones T ≈ {rT.mean():.2f}, S ≈ {rS.mean():.1f}; I = {I:.10f}"

caso("1. Regla del trapecio", c1)
caso("2. Regla de Simpson", c2)
caso("3. Orden de convergencia e integral de Planck", c3)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")

Verificando en: /content/taller_f2py 

[PASS] 1. Regla del trapecio -> exacto para una recta y correcto con sin x
[PASS] 2. Regla de Simpson -> exacto para una cúbica y correcto con sin x
[PASS] 3. Orden de convergencia e integral de Planck -> razones T ≈ 4.00, S ≈ 16.1; I = 6.4939394023

Resultado: 3 de 3 PASS
